# Lost in Handoff: baselines, fault smoke test and pilot on a Colab A100

Runs the study on Qwen3-8B through Ollama, with every run going through `run_eval.py`, 4 at a time on
the GPU. Results go to Google Drive. Every sweep resumes, so rerunning a cell skips trials that already
have a record.

- Sections 1–4b: setup (always run these).
- Sections 5–7: the clean baselines after the researcher fixes (`baseline_v2`, already done).
- Section 8: **fault smoke test**, F1–F4 on task 0. Stops if a fault did not fire.
- Section 9: **pilot**, every built condition on the first 10 tasks. Conditions whose defense is not
  built yet are skipped; rerun the cell after those PRs are merged and only the new ones run.
- Section 10: pilot summary.

**D4/D5 rerun (Nov 27, after PR #9):** run sections 1–4b, then **8b** (sets the old D4 records aside),
**8c** (clean-D4 check run, about 20 min). If 8c prints GO, run **9** (about 3.5 h on an L4), then **10**.

**Before you start:** Runtime → Change runtime type → **A100 GPU** → Save. For the pilot, run sections
1–4b, then 8, 9 and 10 (Runtime → Run all also works: finished baselines are skipped).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Google Drive
Colab asks you to approve Drive access in a pop-up. Approve it yourself. Drive only stores the results
(and, optionally, a code zip), so a Colab disconnect never loses finished runs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/lost-in-handoff'
# v2 = after the researcher fixes; the pre-fix pilot stays in runs/colab_a100/baseline
OUT = f'{DRIVE_DIR}/runs/colab_a100/baseline_v2'
SMOKE_OUT = f'{DRIVE_DIR}/runs/colab_a100/smoke_v2'
FSMOKE_OUT = f'{DRIVE_DIR}/runs/colab_a100/fault_smoke'  # F1-F4 on task 0 (section 8)
PILOT_OUT = f'{DRIVE_DIR}/runs/colab_a100/pilot'         # every condition, first 10 tasks (section 9)
!mkdir -p {OUT} {SMOKE_OUT} {FSMOKE_OUT} {PILOT_OUT} && ls -la {DRIVE_DIR}

## 2. Code: clone the branch from GitHub, fetch τ²-bench (same commit as local)
Set `BRANCH` to the branch to test. If `USE_ZIP` is True, the code comes from
`My Drive/lost-in-handoff/dist/lost-in-handoff-code.zip` instead (for code not pushed yet).

In [ ]:
import os, shutil, subprocess
BRANCH = 'main'  # or a teammate's branch, to test it before merging
USE_ZIP = False
os.chdir('/content')  # a rerun starts inside the folder it deletes; leave it first
shutil.rmtree('/content/lost-in-handoff', ignore_errors=True)
if USE_ZIP:
    ZIP = f'{DRIVE_DIR}/dist/lost-in-handoff-code.zip'
    print('using', ZIP)
    subprocess.run(['unzip', '-q', ZIP, '-d', '/content/lost-in-handoff'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '--branch', BRANCH,
                    'https://github.com/Shreepoorna1903/lost-in-handoff', '/content/lost-in-handoff'], check=True)
assert os.path.exists('/content/lost-in-handoff/pyproject.toml'), 'no pyproject.toml: wrong branch or zip'
%cd /content/lost-in-handoff
if not USE_ZIP:
    print('testing', subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
!git clone -q https://github.com/sierra-research/tau2-bench vendor/tau2-bench
!git -C vendor/tau2-bench checkout -q 57e54143025bec219b5726884651c98eede202e2
!git -C vendor/tau2-bench rev-parse HEAD
!ls

## 3. Python dependencies (uv honors the local tau2 source in pyproject.toml; pip does not)

In [ ]:
!pip install -q uv
!uv pip install --system -q -e ".[dev]"
!python -m pytest -q 2>&1 | tail -1

## 4. Ollama server (4 runs at a time) and Qwen3-8B
`OLLAMA_NUM_PARALLEL=4` lets 4 requests share the loaded model. `OLLAMA_KEEP_ALIVE=-1` keeps it in GPU
memory between calls.

In [ ]:
!apt-get -qq install -y zstd > /dev/null   # the Ollama installer needs zstd; Colab lacks it
!curl -fsSL https://ollama.com/install.sh | sh
import os, subprocess, time, urllib.request
PARALLEL = 4  # 2 on a T4; must match section 4b
env = dict(os.environ, OLLAMA_NUM_PARALLEL=str(PARALLEL), OLLAMA_KEEP_ALIVE='-1')
subprocess.Popen('ollama serve > /content/ollama.log 2>&1', shell=True, env=env)
for _ in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434'); print('Ollama is up'); break
    except Exception:
        time.sleep(2)
else:
    !tail -20 /content/ollama.log
    raise RuntimeError('Ollama did not start; see the log above')
!ollama pull qwen3:8b
!ollama list

## 4b. Run helpers (always run this cell)
`PARALLEL` runs share the one Ollama server: 4 on an A100 or L4, 2 on a T4 (16 GB).

In [ ]:
import glob, json, re, subprocess, time
MODEL, PARALLEL = 'qwen3:8b', 4  # 2 on a T4

def run_condition(cond, out, extra='', log=None):
    """Run one condition with run_eval.py in PARALLEL shards on the one Ollama server."""
    t0 = time.time()
    sink = f'>> {log}' if log else ''
    procs = [subprocess.Popen(
        f'python run_eval.py --condition {cond} --model {MODEL} --out {out} {extra} '
        f'--shard {k}/{PARALLEL} 2>&1 | grep -E --line-buffered "^task|Error|Traceback" {sink}', shell=True)
        for k in range(1, PARALLEL + 1)]
    for p in procs:
        p.wait()
    print(f'{cond}: done in {(time.time() - t0) / 60:.1f} min')

CLAIMED_DONE = re.compile(r'\b(has|have|was|were) been (processed|completed|initiated|made|done|'
                          r'exchanged|returned|cancelled|canceled|modified|refunded)|successfully', re.I)

def provenance_check(cond, out=None, quiet=False):
    """D4/D5: per run, whether the researcher tagged its handoff with [call #N], how often the
    executor used get_tool_call_record (lookups; requeries = D5 re-runs), and whether anything was
    written. Returns the totals."""
    out = out or PILOT_OUT
    rows = []
    for f in sorted(glob.glob(f'{out}/qwen3-8b/{cond}/*.json')):
        r = json.load(open(f))
        if r.get('status') != 'ok':
            print(f"  task {r.get('task_id')}: status={r.get('status')}")
            continue
        h = next((x['content'] for x in r['state']['handoffs'] if x['from'] == 'researcher'), '')
        u = r.get('usage') or {}
        row = {'task': r['task_id'], 'solved': bool(r.get('success')),
               'tags': len(re.findall(r'\[call #\d+\]', h)),
               'lookups': u.get('lookups', 0), 'requeries': u.get('requeries', 0),
               'writes': sum(1 for t in r['state']['tool_log'] if t.get('is_write') and not t.get('error'))}
        rows.append(row)
        if not quiet:
            print(f"  task {row['task']:>3}  solved={row['solved']!s:<5}  tags={row['tags']:<3}  "
                  f"lookups={row['lookups']:<3}  requeries={row['requeries']:<3}  writes={row['writes']}")
    s = {'runs': len(rows), 'solved': sum(r['solved'] for r in rows),
         'tagged': sum(r['tags'] > 0 for r in rows), 'looked_up': sum(r['lookups'] > 0 for r in rows),
         'lookups': sum(r['lookups'] for r in rows), 'requeries': sum(r['requeries'] for r in rows),
         'no_write': sum(r['writes'] == 0 for r in rows)}
    n = s['runs']
    print(f"{cond:<9} runs {n}  solved {s['solved']}/{n}  tagged handoffs {s['tagged']}/{n}  "
          f"runs with a lookup {s['looked_up']}/{n} ({s['lookups']} lookups, {s['requeries']} re-queries)  "
          f"no write {s['no_write']}/{n}")
    return s

## 5. Optional smoke test: the pipeline on 4 tasks that failed in the pilot
Skip this section to go straight to the full baseline. Tasks 0 and 2: the researcher claimed the changes
were already made. Tasks 3 and 13: it hit the turn limit. One trial each (about 5 minutes on an A100).
For each task the cell prints success, whether a wrap-up call fired, any "already done" wording in the
researcher's handoff, and the handoffs. **Check:** no handoff says an action was done, and the executor
makes write calls.

In [ ]:
run_condition('clean-D0', SMOKE_OUT, '--tasks 0,2,3,13')
for f in sorted(glob.glob(f'{SMOKE_OUT}/qwen3-8b/clean-D0/*.json')):
    r = json.load(open(f))
    if r.get('status') != 'ok':
        print(f, 'ERROR', r.get('error')); continue
    hs = r['state']['handoffs']
    res = next((h['content'] for h in hs if h['from'] == 'researcher'), '')
    writes = [t['tool'] for t in r['state']['tool_log'] if t.get('agent') == 'executor']
    print('=' * 100)
    print(f"task {r['task_id']}  success={r['success']}  wrap_ups={r['usage'].get('wrap_ups')}  "
          f"executor calls={writes}")
    print('claims done:', CLAIMED_DONE.findall(res) or 'none')
    print('--- intake handoff:\n', hs[0]['content'][:800])
    print('--- researcher handoff:\n', res[:1500])

## 6. Baseline sweep: single-clean, then clean-D0, 4 shards at a time, resumable
Runs the first `LIMIT` tasks (no-op tasks 10, 12, 50, 105 are already filtered out). Progress goes to
`OUT/sweep.log`. If the session dies, reconnect, rerun sections 1–4b, then rerun this cell.

In [ ]:
LIMIT = 10
LOG = f'{OUT}/sweep.log'
for cond in ['single-clean', 'clean-D0']:
    with open(LOG, 'a') as log:
        log.write(f'=== {cond} {time.strftime("%F %T")}\n')
    run_condition(cond, OUT, f'--limit {LIMIT}', log=LOG)
!tail -25 {LOG}

## 7. Summary

In [ ]:
!python score_eval.py {OUT}/qwen3-8b 2>&1 | grep -v DEBUG
for cond in ['single-clean', 'clean-D0']:
    res = {}
    for f in glob.glob(f'{OUT}/qwen3-8b/{cond}/*.json'):
        r = json.load(open(f))
        res[r['task_id']] = r.get('success')
    print(f'{cond:<14} solved', sorted((t for t, ok in res.items() if ok), key=int),
          ' failed', sorted((t for t, ok in res.items() if not ok), key=int))

## 8. Fault smoke test: F1–F4 on task 0
One run per fault, with no defense (about 10 minutes; F4 takes longer because it does a clean run and
then forks it). For each run the cell prints the status, success, whether the fault fired, and where
the planted value went. **Check:** F1–F3 show `fired=True`; F4 is `ok` (or `no_target` with a reason,
when the researcher never wrote the value). The cell stops with an error otherwise: send the output
before starting the pilot.

In [ ]:
FSMOKE = ['F1-D0', 'F2-D0', 'F3-D0', 'F4-D0']
for cond in FSMOKE:
    run_condition(cond, FSMOKE_OUT, '--tasks 0')

problems = []
for cond in FSMOKE:
    f = f'{FSMOKE_OUT}/qwen3-8b/{cond}/task0__trial0.json'
    try:
        r = json.load(open(f))
    except FileNotFoundError:
        problems.append(f'{cond}: no record'); continue
    st, fs, t = r.get('status'), r.get('faults') or [], r.get('taint') or {}
    fired = all(x.get('fired') for x in fs) if fs else None
    print(f"{cond}  status={st}  success={r.get('success')}  fired={fired}  "
          f"passed_on={t.get('in_handoffs')}  written={t.get('reached_write')}  "
          f"{r.get('no_target_reason', '')}{r.get('error', '')}")
    if st == 'error' or (st == 'ok' and not fired):
        problems.append(f'{cond}: status={st} fired={fired}')
assert not problems, 'fault smoke test failed: ' + '; '.join(problems)
print('\nfault smoke test passed')

## 8b. D4/D5 rerun: set the old D4 records aside
PR #9 changed D4 (the executor is now told to check its sources, and the researcher to tag every ID),
so the 50 D4 runs from before it no longer match the code. This cell **moves** them to `pilot_d4_v2`.
Moving, not copying, matters: section 9 skips any task that already has a record. Safe to rerun: once a
condition is set aside it is never moved again, so new D4 records stay where they are.

In [ ]:
src_dir = f'{PILOT_OUT}/qwen3-8b'
dst_dir = f'{DRIVE_DIR}/runs/colab_a100/pilot_d4_v2/qwen3-8b'
os.makedirs(dst_dir, exist_ok=True)
for c in ['clean-D4', 'F1-D4', 'F2-D4', 'F3-D4', 'F4-D4']:
    if os.path.exists(f'{dst_dir}/{c}'):
        print(f'{c}: already set aside')
    elif os.path.exists(f'{src_dir}/{c}'):
        shutil.move(f'{src_dir}/{c}', f'{dst_dir}/{c}')
        print(f'{c}: moved to pilot_d4_v2')
    else:
        print(f'{c}: nothing to move')
print('pilot now has:', sorted(os.listdir(src_dir)))

## 8c. Check run: clean-D4 on the 10 pilot tasks (about 20 min on an L4)
Before spending 3.5 hours, check that the fix works: the researcher should tag its handoff and the
executor should call `get_tool_call_record`. In the old pilot only 11 of 49 handoffs were tagged and the
executor looked something up in 1 of 49 runs. The cell prints one line per task and then **GO** or
**CHECK**. On CHECK, send the output before running section 9. These runs count: section 9 skips them.

In [ ]:
PLOG = f'{PILOT_OUT}/pilot.log'
commit = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
with open(PLOG, 'a') as log:
    log.write(f'=== clean-D4 check run {time.strftime("%F %T")} commit {commit}\n')
run_condition('clean-D4', PILOT_OUT, '--limit 10', log=PLOG)
s = provenance_check('clean-D4')
go = s['runs'] == 10 and s['tagged'] >= 6 and s['looked_up'] >= 5 and s['solved'] >= 3
print('\nGO: run section 9' if go else '\nCHECK: send this output before running section 9')

## 9. Pilot: every built condition, first 10 tasks, 1 trial each, resumable
Runs the conditions in `src/handoff/conditions.py` in order: `single-clean`, `clean-D0`…`clean-D5`, then
`F1-D0`…`F4-D5` (F4 last, the slowest). A condition whose defense is not built yet is skipped with the
name of its owner. The commit hash goes into `pilot.log` so every record can be tied to the code that
made it. If the session dies, reconnect, rerun sections 1–4b, then rerun this cell: finished runs are
skipped. After the D1, D3 and D5 PRs are merged, rerun sections 2–4b and this cell to run only the
conditions that were missing.

**Nov 27:** D5 is built and D4 changed (PR #9). After 8b and 8c, this cell runs F1–F4-D4, clean-D5 and
F1–F4-D5 (90 runs, about 3.5 h on an L4). Everything else is already done, and D1 and D3 are skipped
until Harsha's PR is merged.

In [ ]:
LIMIT = 10
PLOG = f'{PILOT_OUT}/pilot.log'
commit = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'zip'
# read the condition list in a subprocess, so a re-cloned repo is picked up without restarting the kernel
conds = json.loads(subprocess.run(
    ['python', '-c', 'import json; from handoff.conditions import CONDITIONS as C; '
     'print(json.dumps([[k, v.built, v.owner] for k, v in C.items()]))'],
    capture_output=True, text=True, check=True).stdout)
todo = [c for c, built, _ in conds if built]
waiting = [(c, owner) for c, built, owner in conds if not built]
with open(PLOG, 'a') as log:
    log.write(f'=== pilot {time.strftime("%F %T")} commit {commit}: {len(todo)} conditions to run, '
              f'{len(waiting)} not built\n')
print(f'commit {commit}: running {len(todo)} of {len(conds)} conditions')
for c, owner in waiting:
    print(f'  {c} skipped: not built ({owner})')
for cond in todo:
    with open(PLOG, 'a') as log:
        log.write(f'=== {cond} {time.strftime("%F %T")} commit {commit}\n')
    run_condition(cond, PILOT_OUT, f'--limit {LIMIT}', log=PLOG)
!tail -25 {PLOG}

## 10. Pilot summary
Scores every record in the pilot folder (no model calls). Fault conditions also show `no_target`
(F4 runs with nothing to misreport, left out of the other numbers), `fault_fired`, passed on,
wrong-value write and fault-induced failure against the clean run with the same defense (F2-D4 vs
clean-D4). The second cell shows whether D4 and D5 actually used provenance: tagged handoffs, runs with
a lookup, and D5 re-queries. Report these next to the D4 vs D5 results.

In [ ]:
!python score_eval.py {PILOT_OUT}/qwen3-8b 2>&1 | grep -v DEBUG
done = sorted({os.path.basename(os.path.dirname(f)) for f in glob.glob(f'{PILOT_OUT}/qwen3-8b/*/*.json')})
print(f'\n{len(done)} of 31 conditions have records; missing:',
      [c for c, _, _ in conds if c not in done] or 'none')

In [ ]:
for c in ['clean-D4', 'F1-D4', 'F2-D4', 'F3-D4', 'F4-D4', 'clean-D5', 'F1-D5', 'F2-D5', 'F3-D5', 'F4-D5']:
    if glob.glob(f'{PILOT_OUT}/qwen3-8b/{c}/*.json'):
        provenance_check(c, quiet=True)

## 11. Bring the results home
Download `My Drive/lost-in-handoff/runs/colab_a100/pilot` (and `fault_smoke`, plus `baseline_v2` and
`smoke_v2` if not yet copied) and put them in the project's `runs/colab_a100/` folder, so they can be
scored and committed next to the earlier runs.